# 🔬 基线实验：Linear + XGBoost
**配置**: high complexity, PV+NWP, 24h lookback, 时间编码(TE)

运行顺序：从上到下，一个一个 cell 跑

## 第 0 步：挂载 Google Drive + 安装依赖

In [1]:
# 挂载 Google Drive（会弹出授权窗口，点"允许"即可）
from google.colab import drive
drive.mount('/content/drive')

# 设置项目路径
import os, sys
PROJECT_DIR = '/content/drive/MyDrive/SolarPrediction'
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f'项目路径: {PROJECT_DIR}')
print(f'文件列表: {os.listdir(".")[:10]}...')

Mounted at /content/drive
项目路径: /content/drive/MyDrive/SolarPrediction
文件列表: ['.gitignore', 'WebCrawlerScript.py', 'compare_nwp_single_experiment.py', 'ML+TCN.py', 'FirstRun.1.csv', 'run_single.py', 'unified_run_all.py', 'Inference.ipynb', 'plots.ipynb', 'run_all_experiments.py']...


In [2]:
# 安装缺少的包（Colab 已自带大部分）
!pip install -q xgboost lightgbm rich

In [ ]:
# 检查 GPU 状态
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('✅ XGBoost 将自动使用 GPU 加速')

## 第 1 步：加载数据

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 用绝对路径，避免 kernel reset 后 working dir 丢失
data_path = os.path.join(PROJECT_DIR, 'data', 'Project1140.csv')

t0 = time.time()
df_raw = pd.read_csv(data_path)
df_raw['Datetime'] = pd.to_datetime(df_raw['date'])

print(f'数据大小: {df_raw.shape[0]:,} 行 × {df_raw.shape[1]} 列')
print(f'时间范围: {df_raw["Datetime"].min()} → {df_raw["Datetime"].max()}')
print(f'时间间隔: 10 分钟 (每小时 6 步)')
print(f'加载耗时: {time.time()-t0:.1f} 秒')
df_raw.head()

In [3]:
import os

file_path = os.path.join(PROJECT_DIR, 'data', 'Project1140.csv')
data_dir = os.path.join(PROJECT_DIR, 'data')

print(f'正在检查文件: {file_path}')
if os.path.exists(file_path):
    print('✅ 文件存在！')
else:
    print('❌ 文件不存在。请确认文件路径和文件名是否正确，以及文件是否已上传到 Google Drive 的对应位置。')

print(f'\n`data` 目录内容 ({data_dir}):')
if os.path.exists(data_dir) and os.path.isdir(data_dir):
    for item in os.listdir(data_dir):
        print(f'- {item}')
else:
    print('`data` 目录不存在。')

NameError: name 'PROJECT_DIR' is not defined

## 第 2 步：配置实验参数

In [ ]:
config = {
    'model_complexity': 'high',       # 高复杂度
    'use_pv': True,                   # ✅ 使用历史发电数据
    'use_hist_weather': False,        # ❌ 不用历史天气
    'use_forecast': True,             # ✅ 使用天气预报（NWP）
    'use_ideal_nwp': False,           # ❌ 不用理想天气（不作弊）
    'use_time_encoding': True,        # ✅ 使用时间编码
    'past_hours': 24,                 # 回看 24 小时
    'future_hours': 24,               # 预测未来 24 小时
    'weather_category': 'all_weather',# 使用全部天气特征
    'start_date': '2022-01-01',
    'end_date': '2024-09-28',
}

print('实验配置:')
print(f'  特征组合: PV + NWP（历史发电 + 天气预报）')
print(f'  回看窗口: {config["past_hours"]}h → 预测窗口: {config["future_hours"]}h ({config["future_hours"]*6} 步)')
print(f'  模型复杂度: {config["model_complexity"]}')
print(f'  时间编码: 开启')

## 第 3 步：特征工程
这一步会：
- 清洗数据、筛选日期范围
- 创建时间编码（sin/cos）
- 选择特征（PV历史 + NWP天气预报）
- 创建滑动窗口（把时序数据变成"输入→输出"样本对）

In [ ]:
from data.data_utils import preprocess_features, create_sliding_windows

t0 = time.time()

# 特征预处理
df_clean, hist_feats, fcst_feats, scaler_hist, scaler_fcst, scaler_target, no_hist_power = \
    preprocess_features(df_raw.copy(), config)

print(f'清洗后数据: {df_clean.shape[0]:,} 行')
print(f'历史特征 ({len(hist_feats)}个): {hist_feats}')
print(f'预报特征 ({len(fcst_feats)}个): {fcst_feats}')

# 创建滑动窗口
Xh, Xf, y, hours, dates = create_sliding_windows(
    df_clean, config['past_hours'], config['future_hours'],
    hist_feats, fcst_feats, no_hist_power
)

print(f'\n滑动窗口结果:')
print(f'  历史输入 Xh: {Xh.shape}  (样本数, 时间步, 特征数)')
print(f'  预报输入 Xf: {Xf.shape if Xf is not None else "None"}')
print(f'  预测目标 y:  {y.shape}   (样本数, 预测步数)')
print(f'耗时: {time.time()-t0:.1f} 秒')

## 第 4 步：划分数据集
按时间顺序切分（不能随机打乱，因为是时间序列）

In [ ]:
total = len(Xh)
train_end = int(total * 0.85)

Xh_train, Xh_test = Xh[:train_end], Xh[train_end:]
Xf_train, Xf_test = (Xf[:train_end], Xf[train_end:]) if Xf is not None else (None, None)
y_train, y_test = y[:train_end], y[train_end:]
dates_test = dates[train_end:]

print(f'总样本: {total:,}')
print(f'训练集: {train_end:,} ({train_end/total*100:.0f}%)')
print(f'测试集: {total-train_end:,} ({(total-train_end)/total*100:.0f}%)')

## 第 5 步：训练 Linear（线性回归）基线
最简单的模型，几十秒跑完，作为对比基准

In [ ]:
from train.train_ml import train_ml_model

config_linear = config.copy()
config_linear['model'] = 'Linear'

t0 = time.time()
linear_model, linear_metrics = train_ml_model(
    config_linear, Xh_train, Xf_train, y_train,
    Xh_test, Xf_test, y_test, dates_test,
    scaler_target=scaler_target
)
linear_time = time.time() - t0

print(f'\n✅ Linear 训练完成！耗时: {linear_time:.1f} 秒')
print(f'┌────────────────────────────────────┐')
print(f'│ MAE  = {linear_metrics["mae"]:.4f}  (平均绝对误差)    │')
print(f'│ RMSE = {linear_metrics["rmse"]:.4f}  (均方根误差)      │')
print(f'│ R²   = {linear_metrics["r2"]:.4f}  (决定系数)         │')
print(f'│ sMAPE= {linear_metrics["smape"]:.2f}%                 │')
print(f'└────────────────────────────────────┘')

## 第 6 步：训练 XGBoost
梯度提升树，通常是表格数据上效果最好的模型

In [ ]:
config_xgb = config.copy()
config_xgb['model'] = 'XGB'

t0 = time.time()
xgb_model, xgb_metrics = train_ml_model(
    config_xgb, Xh_train, Xf_train, y_train,
    Xh_test, Xf_test, y_test, dates_test,
    scaler_target=scaler_target
)
xgb_time = time.time() - t0

print(f'\n✅ XGBoost 训练完成！耗时: {xgb_time:.1f} 秒')
print(f'┌────────────────────────────────────┐')
print(f'│ MAE  = {xgb_metrics["mae"]:.4f}                      │')
print(f'│ RMSE = {xgb_metrics["rmse"]:.4f}                      │')
print(f'│ R²   = {xgb_metrics["r2"]:.4f}                        │')
print(f'│ sMAPE= {xgb_metrics["smape"]:.2f}%                   │')
print(f'└────────────────────────────────────┘')

## 第 7 步：对比结果

In [ ]:
print(f'\n{"指标":<12} {"Linear":<12} {"XGBoost":<12} {"谁赢了":<10}')
print(f'{"-"*46}')

for key, name in [('mae','MAE'), ('rmse','RMSE'), ('r2','R²'), ('smape','sMAPE(%)')]:
    lv = linear_metrics[key]
    xv = xgb_metrics[key]
    winner = 'XGBoost' if (xv > lv if key == 'r2' else xv < lv) else 'Linear'
    print(f'{name:<12} {lv:<12.4f} {xv:<12.4f} {winner}')

print(f'\n训练时间:    Linear={linear_time:.1f}s   XGBoost={xgb_time:.1f}s')

# 白天指标对比
print(f'\n--- 白天指标 (10am-6pm) ---')
for key, name in [('daytime_mae','白天MAE'), ('daytime_rmse','白天RMSE'), ('daytime_r2','白天R²')]:
    lv = linear_metrics[key]
    xv = xgb_metrics[key]
    winner = 'XGBoost' if (xv > lv if 'r2' in key else xv < lv) else 'Linear'
    print(f'{name:<12} {lv:<12.4f} {xv:<12.4f} {winner}')

## 第 8 步：画预测对比图

In [ ]:
n_plots = 4
fig, axes = plt.subplots(n_plots, 1, figsize=(14, 12), sharex=True)

t = np.arange(144) * 10 / 60  # 转换为小时

for i in range(n_plots):
    ax = axes[i]
    idx = i * (len(y_test) // n_plots)

    y_true = linear_metrics['y_true'][idx]
    y_pred_linear = linear_metrics['predictions'][idx]
    y_pred_xgb = xgb_metrics['predictions'][idx]

    ax.plot(t, y_true, 'k-', linewidth=1.5, label='实际值 (Ground Truth)')
    ax.plot(t, y_pred_linear, 'b--', linewidth=1.2, alpha=0.8, label='Linear 预测')
    ax.plot(t, y_pred_xgb, 'r--', linewidth=1.2, alpha=0.8, label='XGBoost 预测')

    test_date = dates_test[idx] if idx < len(dates_test) else f'Sample {idx}'
    ax.set_title(f'测试样本 {i+1} ({test_date})', fontsize=11)
    ax.set_ylabel('容量因子 (%)')
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

axes[-1].set_xlabel('预测时间 (小时)')
plt.suptitle('Solar PV 24小时预测对比: Linear vs XGBoost\n(配置: high, PV+NWP, 24h lookback, TE)',
             fontsize=14, fontweight='bold')
plt.tight_layout()

# 保存到 Google Drive
save_path = '/content/drive/MyDrive/SolarPrediction/baseline_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n图片已保存: {save_path}')

## 🎉 实验完成！

**下一步建议：**
- 跑 LSTM 或 TCN 看深度学习是否能超过 XGBoost
- 做消融实验（去掉 NWP / 去掉 TE / 改 lookback）看各因素贡献